# Notebook 4: Run Benchmarks

**Runtime: L4 / T4** (§1 needs no GPU).

## Purpose
Evaluation lives in exactly one place. §1 builds a **leakage-free held-out**
benchmark and records which docs are held out (notebook 2/notebook 3 read this to exclude
them from training). §2+ scans every adapter in `adapters/` plus the zero-shot
base, scores each on the same benchmark, and writes one `results/<id>.json`.
notebook 5 turns those into tables/plots.

## Fixes baked in (from the roadmap)
- **Triage**: report **macro-F1** (classes are imbalanced) + the majority-class baseline.
- **ATT&CK**: reformulated as **constrained multiple-choice** (score candidates),
  not exact-ID generation
- **Held-out**: eval docs are excluded from training via `heldout_ids.json`.

In [1]:
!pip -q install -U "transformers>=4.40.0" "accelerate>=0.28.0" "datasets>=2.18.0" \
  "peft>=0.10.0" "bitsandbytes>=0.43.0" sentencepiece evaluate rouge_score scikit-learn "pandas==2.2.2"


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os, json, math, random, re, glob
import numpy as np, torch
import evaluate as hf_evaluate
from collections import Counter
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

import os, tempfile
def _load_dotenv(path='.env'):
    # Minimal .env loader (no dependency); real environment vars take precedence.
    try:
        with open(path) as _f:
            for _line in _f:
                _line = _line.strip()
                if not _line or _line.startswith('#') or '=' not in _line:
                    continue
                _k, _v = _line.split('=', 1)
                os.environ.setdefault(_k.strip(), _v.strip().strip('\'"'))
    except FileNotFoundError:
        pass
def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False
_load_dotenv()   # pull NVD_API_KEY / FEDDAPT_ROOT from .env if present
if _in_colab():
    from google.colab import drive; drive.mount('/content/drive')
    PROJECT_ROOT = os.environ.get('FEDDAPT_ROOT', '/content/drive/MyDrive/FedDAPT')
else:
    # PyCharm / remote GPU: set FEDDAPT_ROOT to your data dir, e.g.  export FEDDAPT_ROOT=/data/fedapt
    PROJECT_ROOT = os.environ.get('FEDDAPT_ROOT', os.path.abspath('./FedDAPT'))
SCRATCH = os.environ.get('FEDDAPT_WORK', os.path.join(tempfile.gettempdir(), 'fedapt_work'))
os.makedirs(PROJECT_ROOT, exist_ok=True); os.makedirs(SCRATCH, exist_ok=True)
print('PROJECT_ROOT =', PROJECT_ROOT, '| SCRATCH =', SCRATCH)
RAW_DIR   = f'{PROJECT_ROOT}/corpus/curated_clients'
EVAL_DIR  = f'{PROJECT_ROOT}/corpus/eval'; os.makedirs(EVAL_DIR, exist_ok=True)
ADAPTER_DIR = f'{PROJECT_ROOT}/adapters'
RESULTS_DIR = f'{PROJECT_ROOT}/results'; os.makedirs(RESULTS_DIR, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device =', device)

def doc_key(t):
    import hashlib; return hashlib.md5(t.encode('utf-8')).hexdigest()[:12]

## 1. Build the held-out benchmark (NO GPU, run before notebooks 2 & 3)
Holds out a deterministic slice for eval, saves the benchmark + the held-out ids.

In [ ]:
TRIAGE_LEVELS = ['critical', 'high', 'medium', 'low', 'informational']
raw = {os.path.splitext(os.path.basename(p))[0]: json.load(open(p))
       for p in sorted(glob.glob(f'{RAW_DIR}/client_*.json'))}
alld = [d for v in raw.values() for d in v]

# ---- Triage samples: Sigma rules carrying a severity tag ----
triage = []
for t in alld:
    if not t.startswith('Sigma Detection Rule'): continue
    for lv in TRIAGE_LEVELS:
        if f'[{lv.upper()}]' in t:
            prompt = t.split(':', 1)[1].strip() if ':' in t else t   # strip the [LEVEL] leak
            triage.append({'input': prompt, 'label': lv, 'key': doc_key(t)}); break

# ---- ATT&CK samples: technique text -> correct technique id ----
attack = []
for t in alld:
    m = re.match(r'MITRE ATT&CK Technique (T\d+[\.\d]*) \(([^)]+)\)', t)
    if m:
        attack.append({'input': m.group(2), 'label': m.group(1).split('.')[0], 'key': doc_key(t),
                       'text': t.split('):',1)[-1].strip()[:400]})
attack_ids = sorted({a['label'] for a in attack})

# ================= Three-way split: 70% train / 15% val / 15% test =================
RATIOS = (0.70, 0.15, 0.15)
def three_way_split(samples, stratify=True, seed=42):
    rnd = random.Random(seed)
    buckets = {}
    for s in samples:
        buckets.setdefault(s['label'] if stratify else '_', []).append(s)
    tr, va, te = [], [], []
    for items in buckets.values():
        items = items[:]; rnd.shuffle(items)
        n = len(items); n_tr = int(n*RATIOS[0]); n_va = int(n*RATIOS[1])
        tr += items[:n_tr]; va += items[n_tr:n_tr+n_va]; te += items[n_tr+n_va:]
    rnd.shuffle(tr); rnd.shuffle(va); rnd.shuffle(te)
    return tr, va, te

tri_tr, tri_va, tri_te = three_way_split(triage, stratify=True,  seed=42)  # stratify by severity (imbalanced)
atk_tr, atk_va, atk_te = three_way_split(attack, stratify=False, seed=1)   # hundreds of ids -> plain split

json.dump({'triage': tri_va, 'attack': atk_va, 'attack_ids': attack_ids}, open(f'{EVAL_DIR}/val.json',  'w'))
json.dump({'triage': tri_te, 'attack': atk_te, 'attack_ids': attack_ids}, open(f'{EVAL_DIR}/test.json', 'w'))

# ---- Raw-text LM validation set (all doc types) for DAPT perplexity/convergence ----
lm_pool = alld[:]; random.Random(7).shuffle(lm_pool)
lm_val = lm_pool[:min(300, max(1, len(lm_pool)//10))]
json.dump(lm_val, open(f'{EVAL_DIR}/lm_val.json', 'w'))

# ---- Held-out = union of ALL eval docs (val + test + lm_val); notebooks 2 & 3 exclude these ----
heldout_ids = sorted(
    {s['key'] for s in tri_va+tri_te+atk_va+atk_te} | {doc_key(t) for t in lm_val})
json.dump(heldout_ids, open(f'{EVAL_DIR}/heldout_ids.json', 'w'))

def _dist(rows):
    from collections import Counter as _C; return dict(_C(s['label'] for s in rows))
maj = max(_dist(tri_te).values())/len(tri_te) if tri_te else 0
print(f'triage  train/val/test = {len(tri_tr)}/{len(tri_va)}/{len(tri_te)} | test dist {_dist(tri_te)} | majority {maj:.3f}')
print(f'attack  train/val/test = {len(atk_tr)}/{len(atk_va)}/{len(atk_te)} | {len(attack_ids)} techniques')
print(f'LM val: {len(lm_val)} raw docs | held-out ids (excluded from training): {len(heldout_ids)}')

## 2. Model loading + eval functions

In [ ]:
MODEL_ID = 'mistralai/Mistral-7B-v0.1'
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
rouge = hf_evaluate.load('rouge')
VAL  = json.load(open(f'{EVAL_DIR}/val.json'))    # for model selection
TEST = json.load(open(f'{EVAL_DIR}/test.json'))   # for final reporting only

def load_model(adapter_path=None):
    base = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map='auto')
    if adapter_path: base = PeftModel.from_pretrained(base, adapter_path)
    base.eval(); return base

@torch.no_grad()
def score_choice(model, prompt, choices):
    dev = next(model.parameters()).device
    pid = tokenizer(prompt, return_tensors='pt', add_special_tokens=False).input_ids.to(dev)
    best, best_lp = None, -1e9
    for ch in choices:
        cid = tokenizer(' '+ch, return_tensors='pt', add_special_tokens=False).input_ids.to(dev)
        ids = torch.cat([pid, cid], 1)
        lp = torch.log_softmax(model(input_ids=ids).logits, -1)
        s = pid.shape[1]
        tot = sum(lp[0, s+i-1, cid[0, i]].item() for i in range(cid.shape[1])) / cid.shape[1]
        if tot > best_lp: best_lp, best = tot, ch
    return best

def eval_triage(model, bench):
    y, yhat = [], []
    for s in bench['triage']:
        p = f"Classify the severity of this security alert (critical/high/medium/low/informational):\n{s['input']}\nSeverity:"
        yhat.append(score_choice(model, p, TRIAGE_LEVELS)); y.append(s['label'])
    if not y: return {'triage_macro_f1': 0.0, 'triage_acc': 0.0}
    return {'triage_macro_f1': float(f1_score(y, yhat, labels=TRIAGE_LEVELS, average='macro', zero_division=0)),
            'triage_acc': float(np.mean([a == b for a, b in zip(y, yhat)]))}

def eval_attack(model, bench):
    ids = bench['attack_ids']; ok = 0
    for s in bench['attack']:
        distract = random.Random(hash(s['label']) & 0xffff).sample([i for i in ids if i != s['label']], min(7, len(ids)-1))
        choices = distract + [s['label']]; random.shuffle(choices)
        pick = score_choice(model, f"Which ATT&CK technique id best matches: {s['input']}. Answer:", choices)
        ok += int(pick == s['label'])
    return {'attack_mc_acc': float(ok/max(len(bench['attack']), 1)), 'attack_chance': 1.0/min(8, len(ids))}

@torch.no_grad()
def eval_rouge(model, bench, n=50):
    preds, refs = [], []
    for s in bench['attack'][:n]:
        p = f"Summarize this security technique:\n{s['input']}\nSummary:"
        dev = next(model.parameters()).device
        inp = tokenizer(p, return_tensors='pt', truncation=True, max_length=512).to(dev)
        out = model.generate(**inp, max_new_tokens=64, do_sample=False)
        preds.append(tokenizer.decode(out[0], skip_special_tokens=True)[len(p):].strip())
        refs.append(s.get('text', s['input']))
    if not preds: return {'rouge_l': 0.0}
    return {'rouge_l': float(rouge.compute(predictions=preds, references=refs)['rougeL'])}

def full_eval(model, bench):
    return {**eval_triage(model, bench), **eval_attack(model, bench), **eval_rouge(model, bench)}
print('eval ready (VAL for selection, TEST for reporting)')

## 3. Evaluate every adapter (+ zero-shot). Auto-skips finished ones.

In [ ]:
# Registry = zero-shot base + every adapter that has a meta.json
registry = [{'id': 'zeroshot_base', 'stage': 'baseline', 'method': 'zeroshot',
             'epsilon': 'inf', 'mu': 0.0, 'init_from': None, 'client': 'none', 'adapter_path': None}]
for mp in glob.glob(f'{ADAPTER_DIR}/*/meta.json'):
    registry.append(json.load(open(mp)))

print(f'{len(registry)} models to consider')
for meta in registry:
    rid = meta['id']; out = f'{RESULTS_DIR}/{rid}.json'
    if os.path.exists(out): print('skip (done):', rid); continue
    print('evaluating:', rid)
    model = load_model(meta.get('adapter_path'))
    test_m = full_eval(model, TEST)                 # canonical keys = TEST (final report)
    val_m  = full_eval(model, VAL)                  # *_val keys = VALIDATION (model selection)
    rec = {**{k: meta.get(k) for k in ['id','stage','method','epsilon','mu','init_from','client']},
           **test_m, **{f'{k}_val': v for k, v in val_m.items()}}
    json.dump(rec, open(out, 'w'), indent=2)
    del model; torch.cuda.empty_cache()
    print('   test:', {k: round(v,3) for k,v in test_m.items()})
print('done — results in', RESULTS_DIR, '(select best config by *_val, report canonical/test keys)')

## Next
**Notebook 5** (no GPU) reads `results/*.json` into the comparison table and figures.